# Simulated mutation acquisition and maturation accessibility

Panel A moves across the genotype hypercube from germline to fully mature, sampling each step by Kimura fixation probability on the full epistatic landscape and on the additive-only landscape, and retaining each trajectory until its first affinity-decreasing step. The colors each bar is split into provide which substitution fixed at that step.

Panel B sums fixation weights over all monotonic germline-to-mature paths under different antigen-selection schedules, unconditioned and conditioned on a specified intermediate already being present.

Landscapes are reconstructed from the fitted coefficients and floored, which reproduces `predicted_kd_full`; set `scenario_source` to `df_all` to read that column directly instead.

In [ ]:
import os, re, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle, Polygon
from matplotlib.lines import Line2D
from itertools import product
from scipy.sparse import dok_matrix

plt.rcParams["svg.fonttype"] = "path"    # outline text to vectors so Illustrator renders it
plt.rcParams["pdf.fonttype"] = 42        # keep live, editable text in the PDF

## Configuration

In [ ]:
data_dir = "../data"
fig_dir  = "../figures"
os.makedirs(fig_dir, exist_ok=True)
fig_stem = "fig7_simulations"

n_sites = 10                             # mutable sites per antibody -> 2**n_sites genotypes

# Panel A (fixation walk / occupancy)
gamma_traj = 0.5                         # selection scale for the maturation walk
n_pop, n_samples, seed = 1000, 50000, 42

# Panel B (selection-scenario reach)
gamma_scen, n_scen, nrand = 1.0, 1000, 1000
censor_floor = 6.0                       # emulate predicted_kd_full when reconstructing from df_params
scenario_source = "df_params"           # "df_params" (reconstruct + floor) or "df_all" (exact column)

# which antibodies appear in Panel A, and how they group by germline
groups = [("VH1-46 class", ["1NC9", "CH235.12"]),
          ("VH1-2 class", ["N49P7", "VRC01", "VRC07"]),
          ("VRC13", ["VRC13"])]
class_of = {ab: name for name, abs_ in groups for ab in abs_}
row_abs = [["1NC9", "CH235.12", "VRC13"], ["N49P7", "VRC01", "VRC07"]]   # two display rows
antigens = ["CH505TF", "SF162"]

# conditioning columns per antibody in Panel B: (spec, forced-mutation set)
scen_cols = {
    "1NC9": [("unconditioned", []),
             (["S56R", "T57P"], ["S56R", "T57P"]),
             (["N52D", "S53R", "G54R"], ["N52D", "S53R", "G54R"]),
             (["M34I"], ["M34I"])],
    "CH235.12": [("unconditioned", []),
                 (["S56R", "T57P"], ["S56R", "T57P"]),
                 (["E81D"], ["E81D"]),
                 (["M34I"], ["M34I"])],
}

# palette
antigen_colors = {"CH505TF": "#E66100", "SF162": "#5D3A9B"}
traj_palette = ["#0173b2", "#de8f05", "#029e73", "#d55e00", "#cc78bc",
                "#ca9161", "#fbafe4", "#949494", "#ece133", "#56b4e9"]   # seaborn colorblind, one per site
uncond_color, cond_color, mix_color = "#c3cbd6", "#4a5568", "#9aa4b1"
scen_floor = -30                        # left edge of the log-likelihood bars
def tcol(i): return traj_palette[i % n_sites]

# font sizes
group_fs, ab_fs, label_fs, tick_fs, leg_fs = 20, 18, 15, 12, 13.5
axis_label_fs = 19                                            # single consolidated x and y label per display row
b_tick_fs, b_label_fs, b_title_fs, b_cond_fs, b_legend_fs = 14, 18, 21, 15, 16

## Landscape reconstruction

In [ ]:
genotypes = ["".join(t) for t in product("GM", repeat=n_sites)]
geno_masks = np.array([sum(1 << p for p in range(n_sites) if g[p] == "M") for g in genotypes], dtype=np.int64)
def tokens_to_int(g): return sum(1 << (n_sites - 1 - i) for i, ch in enumerate(g) if ch == "M")
geno_ints = np.array([tokens_to_int(g) for g in genotypes], dtype=np.int64)

# display name to file stem; the hyphens and the dot do not appear in any filename
name_to_stem = {"1NC9": "1NC9", "CH235.12": "CH235_12", "N49P7": "N49P7",
                "NIH45-46": "NIH4546", "VRC-PG20": "VRCPG20", "VRC01": "VRC01",
                "VRC07": "VRC07", "VRC13": "VRC13", "VRC16": "VRC16"}

def discover(data_dir):
    # map (antibody, antigen) -> df_params path, and antibody -> meta path
    param_files, meta_files = {}, {}
    for ab in {a for row in row_abs for a in row} | set(scen_cols):
        stem = name_to_stem[ab]
        meta_files[ab] = f"{data_dir}/{stem}_meta.csv"
        for ag in antigens:
            param_files[(ab, ag)] = f"{data_dir}/{stem}_{ag}_BEST_df_params.csv"
    return param_files, meta_files

def load_params(path):
    # pull the intercept and every interaction term (order, site-bitmask, coefficient) out of a fit table
    df = pd.read_csv(path)
    intercept = float(df.loc[df["order"] == 0, "coefficient"].iloc[0])
    terms, max_site = [], -1
    for _, r in df[df["order"] >= 1].iterrows():
        sites = [int(s) for s in re.findall(r"site_(\d+)", str(r["term"]))]
        mask = 0
        for p in sites:
            mask |= 1 << p
            max_site = max(max_site, p)
        terms.append((int(r["order"]), mask, float(r["coefficient"])))
    return intercept, terms, int(df["order"].max()), max_site

def kds_at(intercept, terms, order_cap):
    # evaluate the (uncensored) -log10 Kd landscape, keeping only terms up to order_cap
    vals = np.full(len(geno_masks), intercept)
    for order, mask, coef in terms:
        if order <= order_cap:
            vals[(geno_masks & mask) == mask] += coef
    return vals

def site_meta(path):
    # site labels (germline-AA / Kabat / mature-AA) and regions, ordered site_0..site_9
    m = pd.read_csv(path, dtype={"kabat_position": "str"})
    m = m.assign(_i=m["site"].str.replace("site_", "", regex=False).astype(int)).sort_values("_i")
    labels = [f"{r.germline_aa}{r.kabat_position}{r.mature_aa}" for r in m.itertuples()]
    regions = [str(r.region) for r in m.itertuples()]
    return labels, regions

param_files, meta_files = discover(data_dir)


## Panel A: uphill mutation-order occupancy

Fixation-only walk from germline: at each step a still-available mutation is drawn with probability set by its fixation odds (Kimura, population `n_pop`, selection scale `gamma_traj`). `P_keep` retains only trajectories that have never stepped downhill, so a bar's height is the strictly-uphill surviving fraction and the white gap above it represents the downhill paths excluded.

In [ ]:
def log_fix(sigma, gamma, n):
    # Kimura fixation probability (log) for a variant with selection coefficient sigma in a pop of size n
    x = gamma * sigma
    if np.isclose(x, 0): return -np.log(n)
    if x > 0: return np.log1p(-np.exp(-x)) - np.log1p(-np.exp(-n * x))
    return (n - 1) * x + np.log1p(-np.exp(x)) - np.log1p(-np.exp(n * x))

def step_tables(kds, gamma):
    # per-genotype log fixation rate and destination index for each single forward mutation
    idx = {g: i for i, g in enumerate(genotypes)}
    log_rate = np.full((len(genotypes), n_sites), -np.inf)
    nxt = np.full((len(genotypes), n_sites), -1, dtype=np.int64)
    for i, g in enumerate(genotypes):
        for p in range(n_sites):
            if g[p] != "G": continue
            j = idx.get(g[:p] + "M" + g[p + 1:])
            if j is not None:
                log_rate[i, p] = log_fix(kds[j] - kds[i], gamma, n_pop)
                nxt[i, p] = j
    return log_rate, nxt

def simulate(log_rate, nxt, kds, rng):
    # Gumbel-max sampling of n_samples fixation trajectories; record acquisition order and downhill steps
    start = genotypes.index("G" * n_sites)
    state = np.full(n_samples, start, dtype=np.int64)
    orders = np.full((n_samples, n_sites), -1, dtype=np.int64)
    is_del = np.zeros((n_samples, n_sites), dtype=bool)
    for k in range(n_sites):
        row = log_rate[state]
        active = np.isfinite(row).any(1)
        gumbel = -np.log(-np.log(rng.random((n_samples, n_sites)) + 1e-300) + 1e-300)
        pos = np.where(active, (row + gumbel).argmax(1), -1)
        orders[:, k] = pos
        nx = nxt[state, np.where(active, pos, 0)]
        is_del[:, k] = active & ((kds[np.where(nx >= 0, nx, state)] - kds[state]) < 0)
        state = np.where(active & (nx >= 0), nx, state)
    return orders, is_del

def occupancy(orders, is_del):
    # step x mutation occupancy: p_all over all completed walks, p_keep over strictly-uphill survivors
    keep = ~(orders == -1).any(1)
    ordr, dele = orders[keep], is_del[keep]
    n = int(keep.sum())
    p_all = np.zeros((n_sites, n_sites))
    p_keep = np.zeros((n_sites, n_sites))
    survived = np.cumprod(~dele, axis=1).astype(bool)   # still uphill through step k
    for k in range(n_sites):
        p_all[k] = np.bincount(ordr[:, k], minlength=n_sites) / n
        if survived[:, k].any():
            p_keep[k] = np.bincount(ordr[:, k][survived[:, k]], minlength=n_sites) / n
    return p_all, p_keep

def traj_panel_data(rng):
    # occupancy for every complex at additive (order 1) and full (best order) landscapes
    out = {}
    for ab in [a for row in row_abs for a in row]:
        labels, _ = site_meta(meta_files[ab])
        for ag in antigens:
            intercept, terms, best_order, _ = load_params(param_files[(ab, ag)])
            cell = {"labels": labels, "best": best_order, "orders": {}}
            for order in {1, best_order}:
                kds = kds_at(intercept, terms, order)
                log_rate, nxt = step_tables(kds, gamma_traj)
                cell["orders"][order] = occupancy(*simulate(log_rate, nxt, kds, rng))
            out[(ab, ag)] = cell
    return out


## Panel B: selection-scenario reach

Forward/backward propagation over the genotype hypercube under a per-step antigen exposure schedule. For each conditioning set the bar is the log completion likelihood of reaching the fully mature genotype; the mixed row averages random exposure orderings (mean ± sd over `nrand` draws).

In [ ]:
def nb(s): return bin(s).count("1")                                   # popcount = maturation level
def reachable(s): return [s | (1 << i) for i in range(n_sites) if (s | (1 << i)) > s]

def fix_lin(fs, ft, gamma):
    # linear-fitness fixation probability for a step fs -> ft (guards for numerical extremes)
    d = ft - fs
    if d == 0: return 1.0 / n_scen
    if gamma * d > 100: return 1.0
    if n_scen * gamma * d < -100: return 0.0
    if n_scen * gamma * d > 100: return 1 - np.exp(-gamma * d)
    return (1 - np.exp(-gamma * d)) / (1 - np.exp(-n_scen * gamma * d))

def fit_vector(ab, ag):
    # -log10 Kd for all 2**n_sites genotypes, indexed in the MSB = site 0 space
    if scenario_source == "df_all":
        df = pd.read_csv(f"{data_dir}/{name_to_stem[ab]}_{ag}_BEST_df_all.csv",
                         dtype={"genotype_binary": "str"})
        vals = np.full(2 ** n_sites, np.nan)
        vals[df["genotype_binary"].apply(lambda x: int(x, 2)).values] = df["predicted_kd_full"].values
        return vals
    intercept, terms, best_order, _ = load_params(param_files[(ab, ag)])
    vals = np.empty(2 ** n_sites)
    vals[geno_ints] = kds_at(intercept, terms, best_order)
    return np.maximum(vals, censor_floor)                             # floor emulates the censored column

def build_transition(fit):
    # sparse fixation-transition matrix over the hypercube (row s -> reachable t)
    trans = dok_matrix((2 ** n_sites, 2 ** n_sites))
    for s in range(2 ** n_sites):
        for t in reachable(s):
            trans[s, t] = fix_lin(fit[s], fit[t], gamma_scen)
    return trans.tocsr()

def forward_backward(ctx, trans_by_ag):
    # propagate reach forward from germline and backward from full genotype under exposure schedule ctx;
    # returns per-genotype forward mass and backward completion probability
    f = np.zeros(2 ** n_sites); f[0] = 1.0
    f_list = [f]
    for m in range(n_sites):
        f = f @ trans_by_ag[ctx[m]]
        f_list.append(f)
    b = np.zeros(2 ** n_sites); b[-1] = 1.0
    b_list = [None] * (n_sites + 1); b_list[n_sites] = b
    for m in range(n_sites - 1, -1, -1):
        b = trans_by_ag[ctx[m]] @ b
        b_list[m] = b
    fwd = np.fromiter((f_list[nb(g)][g] for g in range(2 ** n_sites)), float, 2 ** n_sites)
    bwd = np.fromiter((b_list[nb(g)][g] for g in range(2 ** n_sites)), float, 2 ** n_sites)
    return fwd, bwd

def logclip(x):
    # log10 with a floor so zeros/negatives plot at the bar baseline
    x = np.asarray(x, float)
    out = np.full_like(x, -300.0)
    m = x > 0
    out[m] = np.log10(x[m])
    return np.round(out, 2)

def scenario_blocks():
    # the ordered set of single-switch exposure schedules (A^k B^(L-k) and its mirror), de-duplicated
    seq, seen = [], set()
    for a, b in [(0, 1), (1, 0)]:
        for k in range(n_sites + 1):
            blk = tuple([a] * k + [b] * (n_sites - k))
            if blk not in seen:
                seen.add(blk)
                seq.append(list(blk))
    return seq

def scen_panel_data(rng):
    # for 1NC9 and CH235.12: backward completion per scenario, unconditioned reach, and a random-mix summary
    blocks = scenario_blocks()
    out = {}
    for ab in ["1NC9", "CH235.12"]:
        labels, _ = site_meta(meta_files[ab])
        trans_by_ag = [build_transition(fit_vector(ab, ag)) for ag in antigens]
        seq_logb, seq_uncon = [], []
        for blk in blocks:
            fwd, bwd = forward_backward(blk, trans_by_ag)
            seq_logb.append(logclip(bwd).tolist())
            seq_uncon.append(round(float(np.log10(fwd[-1])) if fwd[-1] > 0 else -300.0, 2))
        # Welford mean/sd of log completion across random exposure orderings
        mean = np.zeros(2 ** n_sites); m2 = np.zeros(2 ** n_sites)
        for i in range(nrand):
            _, bwd = forward_backward(rng.integers(0, 2, n_sites), trans_by_ag)
            lc = np.where(bwd > 0, np.log10(np.where(bwd > 0, bwd, 1)), -300.0)
            d = lc - mean; mean += d / (i + 1); m2 += d * (lc - mean)
        out[ab] = {"labels": labels, "seq_logb": seq_logb, "seq_uncon": seq_uncon,
                   "mix_mean": np.round(mean, 2).tolist(),
                   "mix_std": np.round(np.sqrt(m2 / max(nrand - 1, 1)), 2).tolist()}
    return out, blocks


## Plotting

In [ ]:
def stacked_keep(ax, p_keep):
    # one Panel A cell: uphill composition stacked to survival height, downhill left blank white
    mat = np.asarray(p_keep)
    x = np.arange(1, n_sites + 1); bot = np.zeros(n_sites)
    for i in range(n_sites):
        ax.bar(x, mat[:, i], bottom=bot, width=0.82, color=tcol(i), edgecolor="white", lw=0.4)
        bot += mat[:, i]
    ax.set_xticks(x); ax.set_xticklabels([str(v) if v in (1, 6, n_sites) else "" for v in x])
    ax.set_yticks([0.0, 0.5, 1.0])
    ax.set_xlim(0.4, n_sites + 0.6); ax.set_ylim(0, 1.0); ax.tick_params(labelsize=tick_fs)
    for s in ("top", "right"): ax.spines[s].set_visible(False)

def superrow(fig, cell, abs_row, traj):
    # class header, one antibody header spanning each antigen pair, then antigen + order per cell
    complexes = [(ab, ag) for ab in abs_row for ag in antigens]
    gs = cell.subgridspec(5, len(complexes), height_ratios=[0.30, 0.30, 1, 1, 0.60], wspace=0.5, hspace=0.9)
    classes = [class_of[ab] for ab, _ in complexes]
    i = 0
    while i < len(complexes):                       # span a header across each run of same-class columns
        j = i
        while j < len(complexes) and classes[j] == classes[i]:
            j += 1
        axc = fig.add_subplot(gs[0, i:j]); axc.set_xlim(0, 1); axc.set_ylim(0, 1); axc.axis("off")
        axc.text(0.5, 0.42, classes[i], ha="center", va="center", fontsize=group_fs, fontweight="bold")
        axc.plot([0.02, 0.98], [0.0, 0.0], color="#555", lw=1.1)
        i = j
    for idx, ab in enumerate(abs_row):              # antibody label spanning its two antigen columns
        axh = fig.add_subplot(gs[1, 2 * idx:2 * idx + 2]); axh.set_xlim(0, 1); axh.set_ylim(0, 1); axh.axis("off")
        axh.text(0.5, 1.75, ab, ha="center", va="center", fontsize=ab_fs, fontweight="bold", clip_on=False)
    for ci, (ab, ag) in enumerate(complexes):
        c = traj[(ab, ag)]; best = c["best"]
        axf = fig.add_subplot(gs[2, ci]); axd = fig.add_subplot(gs[3, ci])
        stacked_keep(axf, c["orders"][best][1])
        stacked_keep(axd, c["orders"][1][1])
        axf.text(0.5, 1.04, f"{ag}\nfull (order \u2264 {best})", transform=axf.transAxes,
                 ha="center", va="bottom", fontsize=label_fs, fontweight="bold")
        axd.text(0.5, 1.03, "additive (order 1)", transform=axd.transAxes,
                 ha="center", va="bottom", fontsize=label_fs, fontweight="bold")
    # one y label spanning the full+additive pair and one x label spanning all columns
    axy = fig.add_subplot(gs[2:4, 0]); axy.set_xticks([]); axy.set_yticks([]); axy.patch.set_alpha(0)
    for s in axy.spines.values(): s.set_visible(False)
    axy.set_ylabel("fraction of trajectories", fontsize=axis_label_fs, labelpad=24)
    axx = fig.add_subplot(gs[3, :]); axx.set_xticks([]); axx.set_yticks([]); axx.patch.set_alpha(0)
    for s in axx.spines.values(): s.set_visible(False)
    axx.set_xlabel("acquisition step", fontsize=axis_label_fs, labelpad=22)
    for idx, ab in enumerate(abs_row):
        labels = traj[(ab, "CH505TF")]["labels"]
        axl = fig.add_subplot(gs[4, 2 * idx:2 * idx + 2]); axl.axis("off")
        axl.legend(handles=[Line2D([], [], marker="s", linestyle="none", markersize=leg_fs,
                                    markerfacecolor=tcol(k), markeredgecolor="white", markeredgewidth=0.5,
                                    label=labels[k]) for k in range(n_sites)],
                   loc="center", ncol=5, fontsize=leg_fs, frameon=False,
                   handletextpad=0.4, columnspacing=1.1, handlelength=1.0)

def panel_a(fig, cell, traj):
    # Panel A: two display rows of complexes
    gs = cell.subgridspec(2, 1, hspace=0.08)
    superrow(fig, gs[0], row_abs[0], traj)
    superrow(fig, gs[1], row_abs[1], traj)

antigen_hex = [antigen_colors["CH505TF"], antigen_colors["SF162"]]

def _order(blocks):
    # display order for scenario rows: CH505TF-led runs first (longest run on top), then SF162-led
    def runlen(b):
        a = b[0]; k = 0
        for x in b:
            if x == a: k += 1
            else: break
        return a, k
    ch, sf = [], []
    for i, b in enumerate(blocks):
        a, k = runlen(b)
        (ch if a == 0 else sf).append((i, k))
    ch.sort(key=lambda t: -t[1]); sf.sort(key=lambda t: -t[1])
    return [i for i, _ in ch] + [i for i, _ in sf]

def draw_key(ax, blocks, order, show_ylabel, name, start_step, show_xlabel, arrow_step):
    # antigen exposure schedule per scenario row plus the split mixed row; copied in front of each column.
    # horizontal arrow runs from the starting step (its tail) rightward to step 10
    n_blocks = len(blocks); ys = list(range(n_blocks - 1, -1, -1)); y_mix = n_blocks
    for r, si in enumerate(order):
        for j in range(n_sites):
            ax.add_patch(Rectangle((j, ys[r] - .42), 1, .84, color=antigen_hex[blocks[si][j]], lw=.3, ec="white"))
    for j in range(n_sites):                        # mixed row: squares split to read as "either antigen"
        x0, x1, y0, y1 = j, j + 1, y_mix - .42, y_mix + .42
        ax.add_patch(Polygon([(x0, y0), (x0, y1), (x1, y1)], facecolor=antigen_hex[0], edgecolor="white", lw=.3))
        ax.add_patch(Polygon([(x0, y0), (x1, y0), (x1, y1)], facecolor=antigen_hex[1], edgecolor="white", lw=.3))
    ax.set_xlim(0, n_sites); ax.set_ylim(-.6, n_blocks + 2.2)
    if start_step is not None:
        ax.set_xticks([start_step - 0.5, 9.5]); ax.set_xticklabels([str(start_step), "10"], fontsize=b_tick_fs)
    else:
        ax.set_xticks([0.5, 5.5, 9.5]); ax.set_xticklabels(["1", "6", "10"], fontsize=b_tick_fs)
    if show_ylabel:
        ax.set_yticks([y_mix]); ax.set_yticklabels(["mixed"], fontsize=b_tick_fs)
    else:
        ax.set_yticks([])
    if show_xlabel: ax.set_xlabel("step", fontsize=b_label_fs)
    if name: ax.set_title(name, fontsize=b_title_fs, loc="left", pad=28, fontweight="bold")
    if arrow_step is not None:
        ya = y_mix + 1.2
        ax.annotate("", xy=(9.5, ya), xytext=(arrow_step - 0.5, ya),
                    arrowprops=dict(arrowstyle="->", color="black", lw=1.8))
    for s in ax.spines.values(): s.set_visible(False)

def draw_barcol(ax, ab_data, blocks, order, forced_idx, is_uncon):
    # completion likelihood per scenario for one conditioning set; compressed x axis
    n_blocks = len(blocks); ys = list(range(n_blocks - 1, -1, -1)); y_mix = n_blocks
    forced_int = 0 if is_uncon else sum(1 << (n_sites - 1 - k) for k in forced_idx)
    color = uncond_color if is_uncon else cond_color
    vals = ab_data["seq_uncon"] if is_uncon else [ab_data["seq_logb"][si][forced_int] for si in range(n_blocks)]
    clip = lambda v: max(scen_floor, min(0.0, v))
    for r, si in enumerate(order):
        ax.barh(ys[r], clip(vals[si]) - scen_floor, left=scen_floor, color=color, height=.72)
    err = None if is_uncon else ab_data["mix_std"][forced_int]
    ax.barh(y_mix, clip(ab_data["mix_mean"][forced_int]) - scen_floor, left=scen_floor, color=mix_color, height=.72,
            xerr=(None if err is None else [[0], [err]]), error_kw=dict(ecolor="#4a5568", elinewidth=.9, capsize=2))
    ax.set_xlim(scen_floor, 0); ax.set_ylim(-.6, n_blocks + 2.2); ax.set_yticks([])
    ax.set_xticks([-30, -15, 0]); ax.tick_params(labelsize=b_tick_fs)
    ax.grid(axis="x", color="#e2e6ec", lw=.6); ax.set_axisbelow(True)
    for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)

def panel_b(fig, cell, scen, blocks):
    # 1NC9 and CH235.12: each conditioning set is its own key + bars unit, key first
    order = _order(blocks)
    gs = cell.subgridspec(2, 1, hspace=0.85)
    for row, ab in enumerate(["1NC9", "CH235.12"]):
        ab_data = scen[ab]; name_to_site = {m: i for i, m in enumerate(ab_data["labels"])}
        cols = scen_cols[ab]
        gcols = gs[row].subgridspec(1, len(cols), wspace=0.24)
        for c, (spec, forced) in enumerate(cols):
            is_uncon = spec == "unconditioned"
            pair = gcols[0, c].subgridspec(1, 2, width_ratios=[0.62, 1.5], wspace=0.28)
            draw_key(fig.add_subplot(pair[0, 0]), blocks, order, c == 0,
                     ab if c == 0 else None, None if is_uncon else len(forced) + 1, row == 1,
                     1 if is_uncon else len(forced) + 1)
            axb = fig.add_subplot(pair[0, 1])
            draw_barcol(axb, ab_data, blocks, order, [name_to_site[m] for m in forced], is_uncon)
            label = "unconditioned" if is_uncon else "conditioned on\nstarting with\n{" + ", ".join(forced) + "}"
            axb.set_title(label, fontsize=b_cond_fs, loc="left", pad=6, color="#374151")

def build(traj, scen, blocks, out=None):
    # assemble both panels, save png, svg and pdf
    out = out or f"{fig_dir}/{fig_stem}"
    fig = plt.figure(figsize=(17, 24))
    outer = fig.add_gridspec(2, 1, height_ratios=[1.72, 1.0], hspace=0.12,
                             left=0.055, right=0.99, top=0.955, bottom=0.09)
    reserve = lambda cell, g: cell.subgridspec(2, 1, height_ratios=[1, g], hspace=0.0)[1]  # gap for panel legend
    panel_a(fig, reserve(outer[0], 24), traj)
    panel_b(fig, reserve(outer[1], 4), scen, blocks)
    top1 = outer[1].get_position(fig).y1
    fig.legend(handles=[Patch(color=antigen_hex[0], label="CH505TF"), Patch(color=antigen_hex[1], label="SF162"),
                        Patch(color=uncond_color, label="unconditioned"), Patch(color=cond_color, label="conditioned")],
               loc="upper center", ncol=4, frameon=False, fontsize=b_legend_fs, bbox_to_anchor=(0.5, top1 - 0.006))
    b = outer[1].get_position(fig)
    fig.text((b.x0 + b.x1) / 2 + 0.03, b.y0 - 0.05, "total maturation probability (log a. u.)",
             ha="center", fontsize=b_label_fs)
    for ext in ("png", "svg", "pdf"):
        fig.savefig(f"{out}.{ext}", bbox_inches="tight")
    return fig

## Build

In [ ]:
rng = np.random.default_rng(seed)
traj = traj_panel_data(rng)
scen, blocks = scen_panel_data(rng)
fig = build(traj, scen, blocks)
assert open(f"{fig_dir}/{fig_stem}.svg").read().count("<text") == 0, "live text in svg, check svg.fonttype"
fig